# 02 — Document Intelligence smoke test

Uploads a sample PDF, runs `prebuilt-layout`, prints the first text chunk, any tables found, and extracts embedded images — each image is uploaded to Blob and described by GPT-4o vision so it can be retrieved by the RAG layer and shown to users.

**Place** any PDF at `notebooks/sample.pdf` before running.

In [1]:
import sys, os, pathlib
sys.path.insert(0, os.path.abspath('..'))
from src.blob_client import BlobService
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService

PDF_PATH = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF_PATH.exists(), 'Drop a sample.pdf next to this notebook'

blob = BlobService()
blob.ensure_container()
name = 'docmind-test/sample.pdf'
pdf_bytes = PDF_PATH.read_bytes()
blob.upload(name, pdf_bytes, content_type='application/pdf')
url = blob.get_url(name)
print('PDF URL:', url[:80], '…')

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '7b034d44-4876-11f1-82b5-df1619e59047'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '6d89629c-501e-0011-5483-dccf49000000'
    'x-ms-client-request-id': '7b034d44-4876-11f1-82b5-df1619e59047'
    'x-ms-version': 'REDAC

PDF URL: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p …


In [2]:
doc_intel = DocIntelService()
result = doc_intel.extract_pdf(url)
print('pages:', result['pages'])
print('text chunks:', len(result['text_chunks']))
print('tables:', len(result['tables']))
print('--- first 600 chars of page 1 ---')
print(result['text_chunks'][0]['content'][:600] if result['text_chunks'] else '(empty)')

INFO: Document Intelligence analysing https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p
INFO: Request URL: 'https://azdocumentai.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '237'
    'Accept': 'application/json'
    'x-ms-client-request-id': '828bb5c3-4876-11f1-ab7d-df1619e59047'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
INFO: Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-content-type-options': 'REDACTED'
    'x-ms-region': 'REDACTED'
    'Date': 'Tue, 05 May 2026 11

pages: 27
text chunks: 26
tables: 24
--- first 600 chars of page 1 ---
Multi-Agent Research System
Architecture Design & Technical Documentation
Author
Kumar Nishant - Senior Al Engineer
Team
MLEDSI
Manager
Krutika Kansara
Date
April 20, 2026
Version
1.2
Status
Draft - For Review


In [3]:
if result['tables']:
    print(result['tables'][0]['content'][:1000])
else:
    print('No tables detected.')

| Author | Kumar Nishant - Senior Al Engineer |
| --- | --- |
| Team | MLEDSI |
| Manager | Krutika Kansara |
| Date | April 20, 2026 |
| Version | 1.2 |
| Status | Draft - For Review |


In [3]:
result.keys()

dict_keys(['pages', 'text_chunks', 'tables'])

## Extract embedded images and verbalize them via GPT-4o vision

Document Intelligence does not return raw image bytes, so we use PyMuPDF to pull the embedded images out of the PDF, upload each to Blob Storage, and ask GPT-4o vision to describe it. The text descriptions are what RAG indexes; the `image_url` is what the UI renders so users see the picture when they ask about it.

In [4]:
openai = OpenAIService()
images = doc_intel.extract_images(
    pdf_bytes,
    doc_id='docmind-test',
    blob=blob,
    openai=openai,
)
print(f'extracted {len(images)} images')
for img in images[:5]:
    print(f"- page {img['page']}  {img['ext']}  {img['size_bytes']} bytes  -> {img['blob_name']}")

INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/images/page15_img0.png?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '119879'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'a19bedfd-4876-11f1-bc90-df1619e59047'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Tue, 05 May 2026 11:36:06 GMT'
    'ETag': '"0x8DEAA9A7ECCA39F"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '6d89f84e-501e-0011-6583-dccf49000000'
    'x-m

extracted 1 images
- page 15  png  119879 bytes  -> docmind-test/images/page15_img0.png


In [5]:
from IPython.display import Image, Markdown, display

for img in images[:5]:
    display(Markdown(f"### Page {img['page']} — `{img['blob_name']}`"))
    try:
        display(Image(url=img['image_url']))
    except Exception as e:
        print('(could not render inline:', e, ')')
    display(Markdown(f"**Vision description:**\n\n{img['description']}"))

### Page 15 — `docmind-test/images/page15_img0.png`

**Vision description:**

This image is a flowchart depicting a process for handling a user query through various nodes and tasks. Below is a detailed description of the components and their relationships:

1. **User Query**: The starting point of the process.

2. **Planner Node**: Analyzes intent and defines the goal of the query.

3. **Research Intent Classification**: Classifies the intent into categories such as blog, comparative, deep research, or summary.

4. **Structured Plan Creation**: Generates todos with status set to pending.

5. **Task Manager**: Manages the tasks generated.

6. **Todo Selector**: Picks pending tasks for processing.

7. **Delegation Layer**: Distributes tasks to sub-agents.

8. **Sub-Agents (1, 2, N)**: Each sub-agent gathers findings and sources.

9. **Aggregator**: Merges, deduplicates, and structures the findings from sub-agents.

10. **Context Synthesizer**: Further processes the aggregated data.

11. **Todo Tracker**: Marks tasks as completed and stores data.

12. **Todo Checker**: Checks if all todos are completed. If pending todos exist, it loops back to the Todo Selector.

13. **Writer Node**: Generates a draft report.

14. **Validator Node**: Checks the draft report for format issues. If valid, it proceeds; otherwise, it loops back to the Writer Node.

15. **Cleanup Node**: Finalizes the report.

16. **Final Report Output**: The end product of the process.

Additional Notes:
- If a research gap is detected, it loops back to the Todo Selector.
- The process ensures all tasks are completed and validated before finalizing the report.